# 06 — Model Comparison (LightGBM vs CatBoost)

Load JSONL metrics from `models/metrics/`, compare models across playlists
on Brier score, log-loss, ROC-AUC, precision@10.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from paths import MODELS_DIR
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

## 1. Load Metrics

In [ ]:
metrics_dir = MODELS_DIR / "metrics"
jsonl_files = sorted(metrics_dir.glob("*.jsonl")) if metrics_dir.exists() else []
print(f"Found {len(jsonl_files)} metrics files")

all_rows: list[dict] = []
summaries: list[dict] = []

for fpath in jsonl_files:
    for line in fpath.read_text().strip().split("\n"):
        if not line:
            continue
        entry = json.loads(line)
        entry["source_file"] = fpath.name
        if "summary" in entry:
            summaries.append(entry)
        else:
            # Flatten metrics into top-level
            metrics = entry.pop("metrics", {})
            entry.update(metrics)
            all_rows.append(entry)

if all_rows:
    df = pl.DataFrame(all_rows)
    print(f"Total metric entries: {len(df)}")
    print(f"Columns: {df.columns}")
    print(f"\nModes: {df['mode'].unique().to_list()}")
    print(f"Model types: {df['model_type'].unique().to_list()}")
    print(f"Playlists: {df['playlist'].n_unique()}")
    df.head(5)
else:
    print("No metrics data found — run train.py first")
    df = pl.DataFrame()

In [ ]:
# Show summaries
if summaries:
    print("Run summaries:")
    for s in summaries:
        print(f"  {s['source_file']}: {s.get('summary', {})}")

## 2. Filter to Compare Mode

Focus on `--compare` runs where both models were evaluated on the same playlists.

In [ ]:
if not df.is_empty():
    compare_df = df.filter(pl.col("mode") == "compare")
    print(f"Compare entries: {len(compare_df)}")

    if compare_df.is_empty():
        # Fall back to train mode if no compare runs exist
        print("No compare runs found — using train mode entries instead")
        compare_df = df

    # Use the most recent run (by timestamp)
    latest_ts = compare_df["timestamp"].unique().sort(descending=True)[0]
    latest = compare_df.filter(pl.col("timestamp") == latest_ts)
    print(f"Latest run: {latest_ts} ({len(latest)} entries)")
    latest.head()

## 3. Per-Playlist Comparison

In [ ]:
KEY_METRICS = ["brier_score", "log_loss", "roc_auc", "precision_at_10", "f1"]

if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    lgbm = latest.filter(pl.col("model_type") == "lightgbm").sort("playlist")
    cat = latest.filter(pl.col("model_type") == "catboost").sort("playlist")

    # Ensure same playlists
    common = set(lgbm["playlist"].to_list()) & set(cat["playlist"].to_list())
    lgbm = lgbm.filter(pl.col("playlist").is_in(list(common))).sort("playlist")
    cat = cat.filter(pl.col("playlist").is_in(list(common))).sort("playlist")
    playlists = lgbm["playlist"].to_list()

    print(f"Comparing {len(playlists)} playlists")

    for metric in KEY_METRICS:
        if metric not in lgbm.columns:
            continue
        lgbm_vals = lgbm[metric].to_numpy()
        cat_vals = cat[metric].to_numpy()

        fig, ax = plt.subplots(figsize=(max(8, len(playlists) * 0.8), 5))
        x = np.arange(len(playlists))
        width = 0.35
        ax.bar(x - width/2, lgbm_vals, width, label="LightGBM", alpha=0.8)
        ax.bar(x + width/2, cat_vals, width, label="CatBoost", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(playlists, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(metric)
        ax.set_title(f"{metric} — LightGBM vs CatBoost")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Need both model types in the same run for comparison")

## 4. Win/Loss Summary

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    win_loss = []

    for metric in KEY_METRICS:
        if metric not in lgbm.columns:
            continue
        lgbm_vals = lgbm[metric].to_numpy()
        cat_vals = cat[metric].to_numpy()

        # For brier_score and log_loss, lower is better
        # For roc_auc, precision_at_10, f1, higher is better
        lower_better = metric in ("brier_score", "log_loss")

        if lower_better:
            lgbm_wins = int((lgbm_vals < cat_vals).sum())
            cat_wins = int((cat_vals < lgbm_vals).sum())
        else:
            lgbm_wins = int((lgbm_vals > cat_vals).sum())
            cat_wins = int((cat_vals > lgbm_vals).sum())

        ties = len(playlists) - lgbm_wins - cat_wins
        win_loss.append({
            "metric": metric,
            "lgbm_wins": lgbm_wins,
            "catboost_wins": cat_wins,
            "ties": ties,
            "lgbm_mean": round(float(lgbm_vals.mean()), 4),
            "catboost_mean": round(float(cat_vals.mean()), 4),
            "direction": "lower=better" if lower_better else "higher=better",
        })

    wl_df = pl.DataFrame(win_loss)
    print("Win/Loss Summary:")
    wl_df

## 5. Score Distributions (Box Plots)

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    fig, axes = plt.subplots(1, len(KEY_METRICS), figsize=(4 * len(KEY_METRICS), 5))
    if len(KEY_METRICS) == 1:
        axes = [axes]

    for i, metric in enumerate(KEY_METRICS):
        if metric not in latest.columns:
            continue
        data_pd = latest.select(["model_type", metric]).to_pandas()
        sns.boxplot(data=data_pd, x="model_type", y=metric, ax=axes[i])
        axes[i].set_title(metric)
        axes[i].set_xlabel("")

    plt.suptitle("Metric Distributions by Model Type", fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Metric Correlations

Does good Brier → good precision@10?

In [ ]:
if not latest.is_empty():
    available_metrics = [m for m in KEY_METRICS if m in latest.columns]

    for model_type in ["lightgbm", "catboost"]:
        model_df = latest.filter(pl.col("model_type") == model_type)
        if len(model_df) < 3:
            continue

        corr_data = model_df.select(available_metrics).to_pandas()
        corr = corr_data.corr()

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
        )
        ax.set_title(f"Metric Correlation — {model_type}")
        plt.tight_layout()
        plt.show()

## 7. Per-Playlist Deep Dive (Biggest Disagreements)

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    # Find playlists where models disagree most on Brier score
    if "brier_score" in lgbm.columns:
        brier_diff = np.abs(lgbm["brier_score"].to_numpy() - cat["brier_score"].to_numpy())
        top_diff_idx = np.argsort(brier_diff)[::-1][:5]

        print("Playlists with largest Brier score disagreement:")
        for idx in top_diff_idx:
            pname = playlists[idx]
            lgbm_row = lgbm.filter(pl.col("playlist") == pname)
            cat_row = cat.filter(pl.col("playlist") == pname)
            print(f"\n  {pname}:")
            for metric in KEY_METRICS:
                if metric in lgbm_row.columns:
                    lv = lgbm_row[metric][0]
                    cv = cat_row[metric][0]
                    delta = cv - lv
                    print(f"    {metric:20s}  LGBM={lv:.4f}  Cat={cv:.4f}  Δ={delta:+.4f}")

## 8. Temporal Stability (across runs)

In [ ]:
if not df.is_empty():
    timestamps = df["timestamp"].unique().sort().to_list()
    if len(timestamps) > 1:
        print(f"{len(timestamps)} distinct training runs found")

        # Mean Brier per run per model
        run_means = (
            df.filter(pl.col("brier_score").is_not_null())
            .group_by(["timestamp", "model_type"])
            .agg(pl.col("brier_score").mean().alias("mean_brier"))
            .sort("timestamp")
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        for mt in run_means["model_type"].unique().to_list():
            subset = run_means.filter(pl.col("model_type") == mt)
            ax.plot(subset["timestamp"].to_list(), subset["mean_brier"].to_list(), marker="o", label=mt)

        ax.set_xlabel("Run timestamp")
        ax.set_ylabel("Mean Brier Score")
        ax.set_title("Brier Score Stability Across Runs")
        ax.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Only 1 run found — need multiple runs for temporal comparison")

## 9. Recommendation Summary

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    print("=" * 60)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 60)
    print(f"Playlists compared: {len(playlists)}")
    print(f"Run: {latest_ts}")
    print()

    for _, row in wl_df.iter_rows(named=True):
        pass  # just need last

    for row in wl_df.iter_rows(named=True):
        winner = "LightGBM" if row["lgbm_wins"] > row["catboost_wins"] else "CatBoost" if row["catboost_wins"] > row["lgbm_wins"] else "Tie"
        print(f"{row['metric']:20s}  Winner: {winner:10s}  (LGBM {row['lgbm_mean']:.4f} vs Cat {row['catboost_mean']:.4f})")

    print()
    print("Decision criteria:")
    print("  - Brier score (primary): lower = better calibrated probabilities for reranking")
    print("  - Log-loss: lower = better probability estimates")
    print("  - Precision@10: higher = better top-k rerank quality")
    print("  - ROC-AUC: higher = better discrimination")
else:
    print("Insufficient data for summary")